<!-- NB_DOC_INTRO_V1 -->
# Data Science Geospatiale

> 📚 **Doc thematique** : [docs/02_EDA.md](docs/02_EDA.md) (EDA & Visualisation)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

La **donnee geospatiale** (adresses, polygones, trajets, zones) requiert un outillage specifique : **systemes de coordonnees** (CRS), **operations spatiales** (intersection, buffer, distance), **indexation spatiale** (R-tree, H3), **visualisation cartographique** (Leaflet/Mapbox).

Ce notebook couvre la stack Python 2024-2025 avec code execute :
- **geopandas** : DataFrame avec colonne `geometry` (Point, LineString, Polygon)
- **shapely** : geometries 2D et leurs operations (intersect, union, buffer, distance)
- **pyproj** : projections / reprojection CRS
- **folium** : cartes interactives Leaflet
- **h3** (Uber) : hexagonal grid system pour agreger geographiquement
- **osmnx** : extraire des donnees OpenStreetMap (rues, POI)
- **geopy** : geocoding (adresse ↔ lat/lon)

Cas d'usage : zones de chalandise, analyse immobiliere, mobilite, sante publique, fraud detection geo.

**Concept critique : CRS** (Coordinate Reference System).
- **EPSG:4326** (WGS84) : lat/lon en **degres**, **standard mondial** (GPS)
- **EPSG:3857** (Web Mercator) : projection plate en metres (tuiles Google/OSM)
- **EPSG:2154** (Lambert-93) : France metropole en metres
- ⚠️ **Calculer une distance en EPSG:4326 = faux !** Toujours reprojeter en CRS metrique.

## Plan

1. Installation et concepts (CRS, projections)
2. **shapely** : geometries de base
3. **geopandas** : GeoDataFrame, reprojection, jointures spatiales
4. **Distance Haversine** vectorisee (calculs sans reprojection)
5. **H3** (Uber) : hexagonal binning
6. **folium** : cartes interactives (markers, circles, polygons, choropleth)
7. **osmnx** : extraction OpenStreetMap
8. **geopy** : geocoding
9. Cas d'usage : densite par H3, zone de chalandise
10. Pieges et anti-patterns
11. References


## 1. Installation et concepts


In [ ]:
# pip install -q geopandas shapely folium h3 numpy pandas scipy matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd
import shapely
from shapely.geometry import Point, LineString, Polygon, MultiPolygon, box

print(f"geopandas : {gpd.__version__}")
print(f"shapely   : {shapely.__version__}")

## 2. shapely — geometries de base

`shapely` est la lib **fondamentale** : tous les objets geometriques (Point, LineString, Polygon) viennent de la, et geopandas s'appuie dessus.


In [ ]:
# Creation
p1 = Point(2.3522, 48.8566)        # Paris (lon, lat) — ATTENTION ordre !
p2 = Point(4.8357, 45.7640)        # Lyon
ligne = LineString([p1, p2])
polygone = Polygon([(2.2, 48.8), (2.4, 48.8), (2.4, 48.9), (2.2, 48.9)])

print(f"p1.x={p1.x}, p1.y={p1.y}")
print(f"ligne.length (en degres !) : {ligne.length:.4f}")
print(f"polygone.area  (degres²)   : {polygone.area:.4f}")
print(f"polygone.bounds            : {polygone.bounds}")
print(f"p1 within polygone         : {polygone.contains(p1)}")

In [ ]:
# Operations ensemblistes
poly_a = Polygon([(0, 0), (2, 0), (2, 2), (0, 2)])
poly_b = Polygon([(1, 1), (3, 1), (3, 3), (1, 3)])

print(f"poly_a.area              : {poly_a.area}")
print(f"poly_b.area              : {poly_b.area}")
print(f"intersection area        : {poly_a.intersection(poly_b).area}")
print(f"union area               : {poly_a.union(poly_b).area}")
print(f"poly_a - poly_b (diff)   : {poly_a.difference(poly_b).area}")
print(f"poly_a intersects poly_b : {poly_a.intersects(poly_b)}")

In [ ]:
# Buffer (zone autour) — IMPORTANT : unite = celle du CRS (degres ici !)
buf = p1.buffer(0.01)   # ~1 km a Paris (1 degre ~ 111 km, mais varie avec lat)
print(f"Buffer area (degres²) : {buf.area:.6f}")
print(f"Buffer is_polygon     : {buf.geom_type}")

## 3. geopandas — GeoDataFrame


In [ ]:
# Creer un GeoDataFrame from scratch
data = {
    "name": ["Tour Eiffel", "Sacre-Coeur", "Louvre", "Notre-Dame", "Pantheon"],
    "lat":  [48.8584, 48.8867, 48.8606, 48.8530, 48.8462],
    "lon":  [2.2945, 2.3431, 2.3376, 2.3499, 2.3464],
}
df = pd.DataFrame(data)

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326",     # WGS84 (lat/lon en degres)
)
print(f"CRS : {gdf.crs}")
print(gdf)

### 3.1 Reprojection (CRS) — indispensable pour distances metriques

Sur EPSG:4326 (degres), `gdf.distance(...)` renvoie des **degres**, pas des metres. Pour des metres, reprojeter en **Lambert-93** (France) ou **Web Mercator** (3857, approximatif global).


In [ ]:
# Reprojeter en Lambert-93 (CRS metrique pour France)
gdf_l93 = gdf.to_crs("EPSG:2154")
print(f"Nouveau CRS : {gdf_l93.crs}")
print(f"Premieres coords (en metres) :\n{gdf_l93.geometry.head(2)}")

# Distance Tour Eiffel → Sacre-Coeur en metres
dist_m = gdf_l93.geometry[0].distance(gdf_l93.geometry[1])
print(f"\nDistance Tour Eiffel → Sacre-Coeur : {dist_m:.0f} m  ({dist_m/1000:.2f} km)")

### 3.2 Jointure spatiale (`sjoin`)

Pattern courant : pour chaque point client, attribuer la zone administrative qui le contient.


In [ ]:
# Creer 4 "arrondissements" jouet (rectangles)
arrondissements = gpd.GeoDataFrame({
    "nom": ["Nord-Ouest", "Nord-Est", "Sud-Ouest", "Sud-Est"],
    "geometry": [
        box(2.25, 48.85, 2.35, 48.92),   # bbox NO
        box(2.35, 48.85, 2.45, 48.92),   # bbox NE
        box(2.25, 48.78, 2.35, 48.85),   # bbox SO
        box(2.35, 48.78, 2.45, 48.85),   # bbox SE
    ],
}, crs="EPSG:4326")

# Spatial join : pour chaque point, quel polygone le contient ?
joined = gpd.sjoin(gdf, arrondissements, predicate="within", how="left")
joined[["name", "nom"]]

## 4. Distance Haversine vectorisee

Pour calculer des distances **rapidement** sans reprojeter (utile sur 1M points), on utilise la formule **Haversine** directement sur lat/lon.


In [ ]:
def haversine_vec(lat1, lon1, lat2, lon2, R=6371000):
    """Distance Haversine en metres (vectorisee numpy).

    R = rayon Terre (m). Formule precise (a 0.5% pres).
    """
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# Test : Paris-Lyon doit faire ~ 393 km a vol d'oiseau
d = haversine_vec(48.8566, 2.3522, 45.7640, 4.8357)
print(f"Paris-Lyon (Haversine) : {d/1000:.1f} km")

# Vectorise : distance de tous les points a Paris
center = (48.8566, 2.3522)
dist_to_center = haversine_vec(
    gdf["lat"].values, gdf["lon"].values,
    center[0], center[1],
)
print(f"\nDistance de chaque POI au centre de Paris :")
for name, d in zip(gdf["name"], dist_to_center):
    print(f"  {name:15s} : {d:.0f} m")

## 5. H3 — hexagonal grid system (Uber)

H3 = systeme de **cellules hexagonales** a 16 resolutions (0 = continent, 15 = m²). Utile pour :
- Agreger geographiquement sans avoir de polygones administratifs
- Calculer des voisins de maniere uniforme (toujours 6 voisins, contrairement aux carres ou il y a 4 ou 8 selon distance)
- Indexation rapide spatiale


In [ ]:
import h3

# Lat/lon → cell H3 (resolution 9 ~ 174 m de cote)
cell = h3.latlng_to_cell(48.8584, 2.2945, 9)
print(f"Cell H3 (Tour Eiffel, res 9) : {cell}")

# Cell → boundary (polygone)
boundary = h3.cell_to_boundary(cell)
print(f"Boundary (lat,lon) : {boundary[:3]}...")

# Voisins (anneau rayon 1) — toujours 6 + cellule centrale = 7
neighbors = h3.grid_disk(cell, 1)
print(f"Voisins (k=1) : {len(neighbors)} cellules")

# Cellule parent (resolution moindre)
parent = h3.cell_to_parent(cell, 7)
print(f"Cellule parent (res 7) : {parent}")

# Comparer resolutions
for res in [7, 8, 9, 10]:
    c = h3.latlng_to_cell(48.8584, 2.2945, res)
    area_km2 = h3.cell_area(c, unit="km^2")
    print(f"  res={res} : cell={c}, area={area_km2*1e6:.0f} m²")

### 5.1 Cas d'usage : densite de POI par cellule H3


In [ ]:
# Generer 1000 POI aleatoires autour de Paris
np.random.seed(42)
n_poi = 1000
poi_lat = 48.8566 + np.random.randn(n_poi) * 0.05
poi_lon = 2.3522 + np.random.randn(n_poi) * 0.07

# Assigner chaque POI a sa cellule H3 (res 8)
poi_df = pd.DataFrame({"lat": poi_lat, "lon": poi_lon})
poi_df["h3"] = poi_df.apply(lambda r: h3.latlng_to_cell(r["lat"], r["lon"], 8), axis=1)

# Compter
counts = poi_df.groupby("h3").size().sort_values(ascending=False)
print(f"Nb cellules H3 utilisees : {len(counts)}")
print(f"Top 5 cellules par densite :")
print(counts.head())

## 6. folium — cartes interactives Leaflet


In [ ]:
import folium

# Carte centree sur Paris
m = folium.Map(location=[48.8566, 2.3522], zoom_start=13, tiles="OpenStreetMap")

# Markers
for _, row in gdf.iterrows():
    folium.Marker(
        [row["lat"], row["lon"]],
        popup=f"<b>{row['name']}</b>",
        icon=folium.Icon(color="red", icon="info-sign"),
    ).add_to(m)

# Circle (rayon en metres si CRS est metric — folium en WGS84 utilise des metres pour Circle)
folium.Circle(
    [48.8584, 2.2945],
    radius=500,
    color="blue", fill=True, fill_opacity=0.2,
    popup="500m autour Tour Eiffel",
).add_to(m)

# Polygon (les arrondissements)
for _, row in arrondissements.iterrows():
    coords = [[lat, lon] for lon, lat in row.geometry.exterior.coords]
    folium.Polygon(
        coords, color="green", fill=True, fill_opacity=0.1,
        popup=row["nom"],
    ).add_to(m)

# Sauvegarder (HTML interactif)
m.save(str(Path(tempfile.gettempdir()) / "paris_map.html"))
print("Carte sauvee dans /tmp/paris_map.html — l'objet 'm' est aussi affichable en notebook")
# En notebook : m

### 6.1 Heatmap (densite de points)


In [ ]:
from folium.plugins import HeatMap

m_heat = folium.Map(location=[48.8566, 2.3522], zoom_start=12, tiles="CartoDB positron")

# HeatMap attend [(lat, lon, weight), ...] ou [(lat, lon), ...]
heat_data = list(zip(poi_lat, poi_lon))
HeatMap(heat_data, radius=15, blur=10).add_to(m_heat)
m_heat.save(str(Path(tempfile.gettempdir()) / "paris_heatmap.html"))
print("Heatmap sauvee dans", Path(tempfile.gettempdir()) / "paris_heatmap.html")

### 6.2 Choropleth — coloration par zone

Pour colorer des regions selon une valeur (densite, prix moyen, etc.). Necessite un GeoJSON.


In [ ]:
# Choropleth jouet : coloration des arrondissements par nb de POI contenus
arr_with_count = arrondissements.copy()
poi_gdf = gpd.GeoDataFrame(
    poi_df, geometry=gpd.points_from_xy(poi_df["lon"], poi_df["lat"]), crs="EPSG:4326",
)

# Spatial join : compter les POI par arrondissement
joined = gpd.sjoin(poi_gdf, arrondissements, predicate="within", how="inner")
counts_per_arr = joined.groupby("nom").size().reset_index(name="n_poi")
arr_with_count = arr_with_count.merge(counts_per_arr, on="nom", how="left").fillna(0)
print(arr_with_count[["nom", "n_poi"]])

# Choropleth folium
m_chor = folium.Map(location=[48.8566, 2.3522], zoom_start=12)
folium.Choropleth(
    geo_data=arr_with_count.__geo_interface__,
    data=arr_with_count,
    columns=["nom", "n_poi"],
    key_on="feature.properties.nom",
    fill_color="YlOrRd", fill_opacity=0.6, line_opacity=0.5,
    legend_name="Nb POI",
).add_to(m_chor)
m_chor.save(str(Path(tempfile.gettempdir()) / "paris_choropleth.html"))
print("Choropleth sauvee dans", Path(tempfile.gettempdir()) / "paris_choropleth.html")

## 7. osmnx — extraction OpenStreetMap

`osmnx` permet d'extraire le graphe routier d'une ville, faire du routing, calculer des isochrones.

> ⚠️ Necessite une connexion internet (queries Overpass API). Code commente, decommenter pour run.


In [ ]:
# pip install -q osmnx networkx
# import osmnx as ox
# import networkx as nx

# # Graphe routier d'un quartier (~ 1 min en DL)
# G = ox.graph_from_place("Le Marais, Paris, France", network_type="walk")
# print(f"Graph : {len(G.nodes)} nodes, {len(G.edges)} edges")

# # Shortest path entre 2 points
# orig = ox.distance.nearest_nodes(G, X=2.3522, Y=48.8566)
# dest = ox.distance.nearest_nodes(G, X=2.3625, Y=48.8625)
# route = nx.shortest_path(G, orig, dest, weight="length")
# fig, ax = ox.plot_graph_route(G, route, route_color="r", route_linewidth=4)

# # POIs (restaurants)
# pois = ox.features_from_place("Le Marais, Paris, France", tags={"amenity": "restaurant"})
# print(f"{len(pois)} restaurants dans Le Marais")
print("osmnx : decommenter pour run (requiert internet + ~ 1 min DL)")

## 8. geopy — geocoding

Adresse → (lat, lon), et inverse. **Nominatim** (gratuit, OSM) limite a 1 req/sec.


In [ ]:
# pip install -q geopy
# from geopy.geocoders import Nominatim
# from geopy.extra.rate_limiter import RateLimiter

# geolocator = Nominatim(user_agent="my_demo_app")
# # Rate limit obligatoire avec Nominatim (politesse)
# geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# # Geocoding
# loc = geocode("Tour Eiffel, Paris")
# print(f"Tour Eiffel : ({loc.latitude}, {loc.longitude})")

# # Reverse
# loc = geolocator.reverse((48.8584, 2.2945))
# print(f"Adresse : {loc.address}")
print("geopy : decommenter (requiert internet, lent : 1 req/sec via Nominatim)")

## 9. Cas d'usage : zone de chalandise

**Probleme** : pour chaque magasin, identifier la "zone de chalandise" = 15 minutes a pieds + croiser avec densite de population.

**Approche** :
1. Pour chaque magasin → calculer l'**isochrone** (osmnx + networkx)
2. Croiser avec un **GeoJSON de population** (INSEE IRIS)
3. Aggreger : population dans la zone de chalandise par magasin


In [ ]:
# Pseudo-code complet (necessite osmnx + INSEE GeoJSON)
print('''
# 1. Charger les magasins (GeoDataFrame)
stores = gpd.GeoDataFrame(...)

# 2. Pour chaque magasin, calculer isochrone 15 min a pieds
# import osmnx as ox
# import networkx as nx
# G = ox.graph_from_place("Paris", network_type="walk")
# isochrones = []
# for _, store in stores.iterrows():
#     center = ox.distance.nearest_nodes(G, X=store.geometry.x, Y=store.geometry.y)
#     subgraph = nx.ego_graph(G, center, radius=15*60, distance="travel_time")
#     poly = ox.utils_geo.bbox_to_poly(*ox.utils_geo.bbox_from_point((y, x), 1000))
#     isochrones.append(poly)
# stores["isochrone"] = isochrones

# 3. Charger IRIS INSEE (population par cellule)
iris = gpd.read_file("iris.geojson")  # https://geoservices.ign.fr/

# 4. Spatial join : intersection isochrone × iris
overlay = gpd.overlay(stores.set_geometry("isochrone"), iris, how="intersection")

# 5. Pondere par surface intersecte (si IRIS partiellement couvert)
overlay["pop_pondere"] = overlay["population"] * (overlay.geometry.area / overlay.iris_area)

# 6. Sum par magasin
result = overlay.groupby("store_id")["pop_pondere"].sum()
''')

## 10. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Distance en EPSG:4326 (degres) | Reprojeter en CRS metrique (3857 ou local Lambert) avant `.distance()` |
| `gdf.buffer(1000)` en EPSG:4326 | `to_crs(3857)` d'abord, sinon "1000 degres" !! |
| Coords (lat, lon) au lieu de (lon, lat) | Shapely : `Point(lon, lat)`. Folium : `[lat, lon]`. Toujours verifier |
| Charger toute la France en 1 GeoJSON | Format spatial indexe : **GeoParquet**, Geopackage |
| Spam Nominatim sans `RateLimiter` | Obligatoire (1 req/sec), sinon ban |
| Lat/Lon en string ("48.8584") | Convertir explicitement en float |
| Confondre **CRS projete** (metres) et **geographique** (degres) | `.crs.is_projected` |
| Spatial join sans `predicate=` | Toujours specifier : within / intersects / contains / overlaps |
| Lecture shapefile sans CRS | Toujours specifier `crs=` ou verifier `gdf.crs is not None` |


## 11. References

### Documentation officielle
- **geopandas** : https://geopandas.org/
- **shapely** : https://shapely.readthedocs.io/
- **folium** : https://python-visualization.github.io/folium/
- **h3-py** : https://uber.github.io/h3-py/ , concepts : https://h3geo.org/
- **osmnx** : https://osmnx.readthedocs.io/
- **geopy** : https://geopy.readthedocs.io/
- **pyproj** : https://pyproj4.github.io/pyproj/

### Datasets publics (France)
- **IGN BD TOPO** : https://geoservices.ign.fr/
- **INSEE IRIS** : 50k zones, ~ 2000 habitants chacune
- **OpenStreetMap** via osmnx
- **data.gouv.fr** : equipements, transports

### Voir aussi (collection)
- [EDA_Visualisation_Introduction.ipynb](./EDA_Visualisation_Introduction.ipynb) — viz generique
- [BDD_DuckDB.ipynb](./BDD_DuckDB.ipynb) — SQL spatial avec extension `spatial`
